In [14]:
import json
from pathlib import Path

from tqdm.auto import tqdm

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans

Top 10 most frequent themes

In [ ]:

df = pd.read_csv('/Users/mariem/Desktop/docs_with_themes.csv')

# Filter only captions (not images)
captions = df[df['kind']=="caption"]

# Count occurrences by theme_id
theme_counts = captions['theme_id'].value_counts().head(10)
theme_counts


theme_id
1     50119
17    42959
58    41912
31    41156
43    40476
49    40263
61    39914
54    38205
76    37890
64    36910
Name: count, dtype: int64

In [5]:
top_ids = [1,17,58,31,43,49,61,54,76,64]

df[df["theme_id"].isin(top_ids)][["theme_id", "theme_label"]].drop_duplicates().sort_values("theme_id")


,theme_id,theme_label
90,1,"Told, Said, Don"
481,17,"Just, Like, Think"
2,31,"Think, Said, Just"
40,43,"Doctor, Effects, Doc"
21,49,"Ll, Let, Come"
828,54,"Just, Told, Don"
62,58,"Species, Extinct, Bird"
104,61,"Said, Just, Think"
237,64,"Gluten Free, Gluten, Mustard"
17,76,"Look Like, Look, Like"


Let's start with cleaning the raw csv files using spacy

In [4]:
import spacy
nlp = spacy.load("en_core_web_sm")

stopwords = nlp.Defaults.stop_words
print(len(stopwords))
print(list(stopwords)[:50])  # preview first 50
#check if like is in stopwords
print("don't" in stopwords)

326
['meanwhile', 'there', 'one', 'seeming', 'her', 'beforehand', 'us', 'for', 'ever', 'have', 'three', 'as', 'so', 'upon', 'behind', 'the', 'somehow', 'again', 'quite', 'top', 'whose', 'they', 'say', 'on', 'amongst', 'together', 'six', 'nowhere', 'against', 'nobody', 'its', '’ve', 'another', 'does', 'whereafter', 'ten', 'was', 'whence', 'elsewhere', 'then', 'whenever', 'their', 'sometime', 'by', '‘d', 'i', 'noone', 'namely', 'anywhere', 'make']
False


Check if recurrent words in our top themes are in the library

In [ ]:

words_to_check = ["like", "don't", "don", "just", "i", "ll", "im", "kinda", "sorta"]

for w in words_to_check:
    doc = nlp(w)
    token = doc[0]
    print(f"{w:10} -> is_stop={token.is_stop}, lemma={token.lemma_}")


like       -> is_stop=True, lemma=like
don't      -> is_stop=True, lemma=do
don        -> is_stop=True, lemma=don
just       -> is_stop=True, lemma=just
i          -> is_stop=True, lemma=I
ll         -> is_stop=True, lemma=ll
im         -> is_stop=True, lemma=I
kinda      -> is_stop=True, lemma=kinda
sorta      -> is_stop=True, lemma=sorta


So It seems we have to add a few stop words 


In [8]:
custom_stopwords = {
    "like", "don", "ll", "kinda", "sorta",
}

for w in custom_stopwords:
    nlp.Defaults.stop_words.add(w)


In [10]:

def clean_caption(text):
    """
    Clean a caption by:
    - running spaCy
    - removing stop words and non-content words
    - keeping only alphabetic NOUN/PROPN/ADJ/VERB
    - lemmatizing and lowercasing
    """
    doc = nlp(str(text))  # str() in case of NaN
    tokens = []
    for token in doc:
        if (
            token.is_alpha
            and not token.is_stop
            and token.pos_ in {"NOUN", "PROPN", "ADJ", "VERB"}
        ):
            tokens.append(token.lemma_.lower())
    return " ".join(tokens)



In [15]:
# General variables: paths
DATA_ROOT = Path("../../data/newyorker_caption_contest")
CONTESTS_JSON = DATA_ROOT / "contests.json"
DATA_DIR = DATA_ROOT / "data"

In [16]:
def load_contests(contests_json_path: Path):
    with open(contests_json_path, "r", encoding="utf-8") as f:
        contests = json.load(f)
    return contests

def build_docs_dataframe():
    contests = load_contests(CONTESTS_JSON)
    docs = []

    # tqdm over contests
    for contest in tqdm(contests, desc="Step 1: data loading/building"):
        cid = contest["contest_id"]

        raw_meta = contest.get("metadata")
        meta = raw_meta or {}   # normalize None -> {}

        # ---- Build one aggregated "image document" per contest ----
        parts = []

        for key in ["image_descriptions", "image_uncanny_descriptions",
                    "image_locations", "questions"]:
            values = meta.get(key) or []
            parts.extend(values)

        entities = meta.get("entities") or []
        for url in entities:
            last = str(url).rstrip("/").split("/")[-1]
            parts.append(last.replace("_", " "))

        image_text = " ".join(map(str, parts)).strip()

        if image_text:
            docs.append(
                {
                    "doc_id": f"image_{cid}",
                    "contest_id": cid,
                    "kind": "image",
                    "text": image_text,
                }
            )

        # ---- Add caption documents for this contest ----
        csv_path = DATA_DIR / f"{cid}.csv"
        if not csv_path.exists():
            print(f"Warning: CSV for contest {cid} not found at {csv_path}")
            continue

        captions_df = pd.read_csv(csv_path)

        # optional inner tqdm if you want per-caption progress too
        for idx, row in captions_df.iterrows():
            cap_text = str(row.get("caption", "")).strip()
            if not cap_text:
                continue

            docs.append(
                {
                    "doc_id": f"caption_{cid}_{idx}",
                    "contest_id": cid,
                    "kind": "caption",
                    "text": cap_text,
                    "rank": row.get("rank", None),
                    "mean": row.get("mean", None),
                    "precision": row.get("precision", None),
                    "votes": row.get("votes", None),
                    "not_funny": row.get("not_funny", None),
                    "somewhat_funny": row.get("somewhat_funny", None),
                    "funny": row.get("funny", None),
                }
            )

    docs_df = pd.DataFrame(docs)
    return docs_df


In [17]:
docs_df = build_docs_dataframe()

Step 1: data loading/building:   0%|          | 0/384 [00:00<?, ?it/s]

In [18]:

tqdm.pandas()   # allows progress bar inside apply()

docs_df["clean_text"] = docs_df["text"].progress_apply(clean_caption)


  0%|          | 0/2292901 [00:00<?, ?it/s]

In [19]:
docs_df[["text", "clean_text"]].head(10)


,text,clean_text
0,A man is relaxing on a city street. Others are...,man relax city street go business step pass ma...
1,I'm a congressman--obstruction is my job.,congressman obstruction job
2,"I'm what they mean when they say, 'The middle ...",mean middle class step
3,Does this suit make me look flat?,suit look flat
4,"When the right woman comes along, I'll know it.",right woman come know
5,"I used to lie in the gutter, but then I quit d...",lie gutter quit drinking
6,"If you can meditate here, you can meditate any...",meditate meditate
7,"Sorry for the inconvenience, I probably should...",inconvenience jump high window
8,I've decided to take it lying down.,decide lie
9,Anyone who steps on me is out!,step


In [20]:
docs_df[docs_df["kind"] == "caption"][["text", "clean_text"]].sample(10)


,text,clean_text
1135649,One of your nuts is defective.,nut defective
1199266,True—but think of the money you’ll save on dry...,true think money save dry cleaning
2175305,And you call me a silly goose,silly goose
1409129,"Like every caveman before you, Jerome, you sho...",caveman jerome learn ask direction
506737,I play all today's hottest tunes.,play today hot tune
1462154,Does anyone want to talk about the elephant in...,want talk elephant room
1104990,Let Zeppelin see THIS!,let zeppelin
113817,You're gonna need a bigger car.,go need big car
697250,After I got the order from Amazon I thought I'...,get order amazon think
639327,"If you think that drop is alarming, wait until...",think drop alarming wait entree


In [22]:

output_path = "/Users/mariem/Desktop/docs_with_clean_text.csv"
docs_df.to_csv(output_path, index=False)
